In [5]:
from dotenv import load_dotenv
load_dotenv()
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

In [6]:
#fetch data from website
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
print(courses_raw)
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)
print(documents[0])
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 84}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]
{'id': '9e508f2212', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: When does

In [13]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course FAQ",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query"
                }
            },
            "required": ["query"]
        }
    }
}


=== iteration #1 ===

FUNCTION CALL: search
ARGS: {"query":"queen gambit"}

=== iteration #2 ===

ASSISTANT:
This search does not return any relevant results. 

The question seems like it may be off-topic. Could you please ask another question about the course, and I will do my best to help?


In [14]:
import json


def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )


def make_call(tool_call):
    args = json.loads(tool_call.function.arguments)

    if tool_call.function.name == "search":
        result = search(**args)
    else:
        raise ValueError(f"Unknown tool: {tool_call.function.name}")

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(result, indent=2)
    }


def agent_loop(
    instructions,
    question,
    model="llama-3.1-8b-instant"
):
    messages = [
        {
            "role": "system",  
            "content": instructions
        },
        {
            "role": "user",
            "content": question
        }
    ]

    iteration = 1
    last_answer = ""

    while True:
        print(f"\n=== iteration #{iteration} ===")

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool],
            tool_choice="auto"
        )

        assistant_message = response.choices[0].message

        # Add assistant message to conversation
        messages.append(
            {
                "role": "assistant",
                "content": assistant_message.content,
                "tool_calls": assistant_message.tool_calls
            }
        )

        # Final answer => no tool calls
        if not assistant_message.tool_calls:
            last_answer = assistant_message.content

            print("\nASSISTANT:")
            print(last_answer)

            break

        # Execute requested tools
        for tool_call in assistant_message.tool_calls:

            print(
                "\nFUNCTION CALL:",
                tool_call.function.name
            )

            print(
                "ARGS:",
                tool_call.function.arguments
            )

            tool_result = make_call(tool_call)

            messages.append(tool_result)

        iteration += 1

    return last_answer

In [15]:
agent_loop(instructions, "what's queen gambit?")


=== iteration #1 ===

FUNCTION CALL: search
ARGS: {"query":"queen gambit"}

FUNCTION CALL: search
ARGS: {"query":"gambit queen gambit"}

FUNCTION CALL: search
ARGS: {"query":"gambit"}

FUNCTION CALL: search
ARGS: {"query":"course logistics queen gambit"}

FUNCTION CALL: search
ARGS: {"query":"queen gambit course"}

=== iteration #2 ===

ASSISTANT:
It appears that the question about the "Queen Gambit" is off-topic. There is no information about the Queen Gambit in the course FAQ.

Is there any other area that you would like to explore?


'It appears that the question about the "Queen Gambit" is off-topic. There is no information about the Queen Gambit in the course FAQ.\n\nIs there any other area that you would like to explore?'